# MCP-backed memory demo

Same agent surface as `memory_skills_demo.ipynb`, but `ctx.memory` is an `MCPMemoryProvider` talking to a Python MCP server (`examples/mcp_memory_server.py`) over stdio. Use this layout when the underlying memory store is owned by a separate service (e.g. Postgres behind an MCP server) and you don't want to expose raw SQL to the agent.

Streamed via `agent.run_stream(...)` + `stream_events(...)` so you can watch MCP `resources/read` and `tools/call` traffic interleave with the agent's tool calls and streamed token output.

Requires `OPENAI_API_KEY` in the environment, the `mcp` optional dependency (`pip install grasp-agents[mcp]`), and `python` on the `PATH`.

**Architectural alignment with the local-FS demo.** Memory authoring goes through the **generic file tools** (`Read` / `Write` / `Edit`). The local demo's toolkit uses `LocalFileBackend`; this one swaps in `MCPFileBackend` (built by `provider.make_file_toolkit()`). The agent code is identical.

The MCP server speaks two surfaces:
* **Resources** (`resources/read`) — the canonical read path. `MEMORY.md` plus each topic file is exposed as `file://<path>`.
* **Tools** (`write_file` / `stat_file` / `delete_file` / `list_dir`) — the mutating + metadata operations. These are what `MCPFileBackend` drives behind the scenes.

**Why one big code cell?** The MCP stdio client uses an anyio cancel scope tied to the asyncio task that opened it. Jupyter runs each cell as a fresh task, so a `connect()` in one cell + a `close()` in another raises. The whole demo lives inside one `async with MCPClient(...)` block; narrative is in the markdown cells above and below.

## Imports + setup

In [ ]:
from pathlib import Path
import shutil
import sys
import tempfile

import grasp_agents
from grasp_agents import (
    LLMAgent,
    MCPClient,
    MCPMemoryProvider,
    MCPServerStdio,
    ProcPacketOutEvent,
    RunContext,
    stream_events,
)
from grasp_agents.llm_providers.openai_completions.completions_llm import (
    OpenAILLM,
    OpenAILLMSettings,
)


def make_llm() -> OpenAILLM:
    return OpenAILLM(
        model_name="openai/gpt-4o-mini",
        llm_settings=OpenAILLMSettings(temperature=0.0),
    )


async def run_and_capture(agent: LLMAgent, message: str, ctx: RunContext):
    final = None
    async for event in stream_events(
        agent.run_stream(message, ctx=ctx),
        show_input_messages=True,
    ):
        if isinstance(event, ProcPacketOutEvent):
            final = event.data.payloads[0]
    return final


TMPDIR = Path(tempfile.mkdtemp(prefix="mcp_memory_demo_"))
MEMDIR = TMPDIR / "memdir"
MEMDIR.mkdir()

(MEMDIR / "MEMORY.md").write_text(
    "# grasp-agents demo memory index\n\n"
    "Topics:\n\n"
    "- [user_preferences](user_preferences.md) \u2014 how the user likes their answers formatted.\n",
    encoding="utf-8",
)
# The body of `user_preferences` lives ONLY in user_preferences.md
# (served via the MCP server). MEMORY.md just links to it — the
# agent reaches the body by `Read`-ing the file (routed through
# MCP `resources/read` by the backend).
(MEMDIR / "user_preferences.md").write_text(
    "---\nname: user_preferences\ntype: user\n"
    "description: how the user likes their answers formatted\n---\n"
    "Reply in concise bullet points unless the user asks for prose.\n",
    encoding="utf-8",
)

SERVER_SCRIPT = Path(grasp_agents.__file__).parent / "examples" / "mcp_memory_server.py"
print("memdir:", MEMDIR)
print("server script:", SERVER_SCRIPT)
print("files:", sorted(p.name for p in MEMDIR.iterdir()))

## End-to-end run

Steps inside the one `async with` block:

1. Spawn the MCP server + connect → `MCPMemoryProvider`.
2. Inspect the snapshot (entries from `resources/list`, lazy `fetch_body` via `resources/read`).
3. Run the agent **read-only** — `tools=None` for structured output; the system prompt has the MEMORY.md index.
4. Run the agent **with the file toolkit** wired via `provider.make_file_toolkit()`. The agent uses `Write` / `Edit` to author a topic file and update `MEMORY.md`; under the hood those calls become MCP `tools/call write_file` and `tools/call stat_file` plus `resources/read` for the pre-edit `Read`.

The MCP server's two surfaces (resources + file tools) make this work end-to-end without exposing any specialized memory tools to the agent.

In [ ]:
async with MCPClient(
    "mem-server",
    server=MCPServerStdio(
        command=sys.executable,
        args=[str(SERVER_SCRIPT), str(MEMDIR)],
    ),
) as client:
    print("connected:", client.name)
    print("discovered tools:", [t.name for t in client.tools()])

    provider = MCPMemoryProvider(
        client=client,
        root=str(MEMDIR),
    )

    # --- 2. Inspect the snapshot ---
    snap = await provider.load()
    print("\nindex (first 80 chars):")
    print(snap.index[:80] if snap.index else snap.index)
    print("\nentries:")
    for e in snap.entries:
        print(f"  {e.name}  type={e.memory_type}  uri={e.uri}")
    print("\nfetch_body('user_preferences'):")
    print(await provider.fetch_body("user_preferences"))

    # --- 3. Agent reads MEMORY.md + a linked topic ---
    print("\n=== Step 1 \u2014 Read tool wired (no writes) ===")
    # provider.make_file_toolkit() returns a FileEditToolkit backed
    # by MCPFileBackend — same tool surface as the local-FS demo,
    # but reads route through MCP `resources/read` under the hood.
    toolkit = provider.make_file_toolkit()
    agent = LLMAgent[str, str, None](
        name="demo",
        llm=make_llm(),
        enable_memory=True,
        # Read-only step: wire Read alone so the agent can fetch
        # the topic body that MEMORY.md points to.
        tools=[toolkit.read],
        sys_prompt="You are a helpful assistant.",
        env_info=False,
        stream_llm=True,
    )
    print("wired tools:", list(agent.tools))
    ctx: RunContext[None] = RunContext(state=None, memory=provider)
    final = await run_and_capture(
        agent,
        "How do I prefer my answers formatted? Look it up if you need to.",
        ctx,
    )
    print("\nfinal:", final)

    # --- 4. Agent authors a memory through MCP file tools ---
    print("\n=== Step 2 \u2014 author via Read/Write/Edit (MCP-backed) ===")
    agent = LLMAgent[str, str, None](
        name="demo",
        llm=make_llm(),
        enable_memory=True,
        tools=toolkit.tools(),
        sys_prompt="You are a helpful assistant.",
        env_info=False,
        stream_llm=True,
    )
    ctx = RunContext(state=None, memory=provider)
    final = await run_and_capture(
        agent,
        "Please remember: I'm based in Berlin (CET timezone).",
        ctx,
    )
    print("\nfinal:", final)

    # The provider caches the snapshot frozen-style — refresh
    # to pick up the topic file the agent just authored.
    await provider.refresh()
    snap = await provider.load()
    print("\nMCP snapshot entries now:", [e.name for e in snap.entries])

print("\nfiles on disk after the run:", sorted(p.name for p in MEMDIR.iterdir()))
print("\nMEMORY.md after the run:")
print((MEMDIR / "MEMORY.md").read_text())

## Cleanup

The MCP server subprocess was terminated when the `async with` block exited. Remove the tmpdir.

In [ ]:
shutil.rmtree(TMPDIR, ignore_errors=True)
print("removed:", TMPDIR)